In [1]:
import os
import sys
from pathlib import Path
import pwd
%pwd

'/home/sheshpal/Documents/ml_llmops_project/NLP_End-to_end[4]_Project/NLP_End_to_End_Project/research'

In [2]:

os.chdir("../")
%pwd

'/home/sheshpal/Documents/ml_llmops_project/NLP_End-to_end[4]_Project/NLP_End_to_End_Project'

In [3]:
from dataclasses import dataclass

@dataclass
class DataTransformationConfig:
    root_dir:Path
    data_path:Path
    tokenizer_path:Path

In [4]:
from  src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directore


In [5]:
class ConfigurationManager:
    def __init__(self,
                 config_file_path = CONFIG_FILE_PATH,
                 params_file_path= PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        
        create_directore([self.config.artifact_root])
        
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        
        create_directore([config.root_dir])
        
        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            tokenizer_path=Path(config.tokenizer_name)
        )
        
        return data_transformation_config

In [6]:
import os
from src.textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk


/home/sheshpal/anaconda3/envs/ml_llmops/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-11-02 08:19:26,854: INFO: utils: NumExpr defaulting to 4 threads.]
[2026-11-02 08:19:27,383: INFO: config: TensorFlow version 2.22.0rc0 available.]


### Data Transformation Component

In [7]:
class DataTransformation:
    def __init__(self, config:DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        
        
    def convert_examples_to_features(self,example_batch):

        input_encodings = self.tokenizer(
                        example_batch['dialogue'],
                        max_length=1024,
                        truncation=True
                                        )

        target_encodings = self.tokenizer(
                            text_target=example_batch['summary'],
                            max_length=128,
                            truncation=True
        
                                        )

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
         }


            
    def convert(self):
        try:
            dataset_samsum = load_from_disk(self.config.data_path)
            dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched = True)
            dataset_samsum.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))
            return dataset_samsum_pt
        except Exception as e:
            raise e

In [8]:
config = ConfigurationManager()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)
data_transformation.convert()

[2026-11-02 08:19:27,728: INFO: common: yaml file: config/config.yaml loadded successfully]
[2026-11-02 08:19:27,731: INFO: common: yaml file: params.yaml loadded successfully]
[2026-11-02 08:19:27,732: INFO: common: created directory at: artifacts]
[2026-11-02 08:19:27,736: INFO: common: created directory at: artifacts/data_transformation]
[2026-11-02 08:19:28,445: INFO: _client: HTTP Request: POST https://huggingface.co/oauth/token "HTTP/1.1 200 OK"]
[2026-11-02 08:19:28,859: INFO: _client: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"]
[2026-11-02 08:19:29,271: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail "HTTP/1.1 200 OK"]
[2026-11-02 08:19:29,611: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-11-02 08:19:30,084: INFO: _client: HTTP Request: HEAD https://huggi

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 87524.00 examples/s] 


DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
})